In [0]:
df = spark.table("yellow_tripdata_2025_01")
display(df.limit(10))

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
1,2025-01-01T00:18:38.000Z,2025-01-01T00:26:59.000Z,1,1.6,1,N,229,237,1,10.0,3.5,0.5,3.0,0.0,1.0,18.0,2.5,0.0,0.0
1,2025-01-01T00:32:40.000Z,2025-01-01T00:35:13.000Z,1,0.5,1,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
1,2025-01-01T00:44:04.000Z,2025-01-01T00:46:01.000Z,1,0.6,1,N,141,141,1,5.1,3.5,0.5,2.0,0.0,1.0,12.1,2.5,0.0,0.0
2,2025-01-01T00:14:27.000Z,2025-01-01T00:20:01.000Z,3,0.52,1,N,244,244,2,7.2,1.0,0.5,0.0,0.0,1.0,9.7,0.0,0.0,0.0
2,2025-01-01T00:21:34.000Z,2025-01-01T00:25:06.000Z,3,0.66,1,N,244,116,2,5.8,1.0,0.5,0.0,0.0,1.0,8.3,0.0,0.0,0.0
2,2025-01-01T00:48:24.000Z,2025-01-01T01:08:26.000Z,2,2.63,1,N,239,68,2,19.1,1.0,0.5,0.0,0.0,1.0,24.1,2.5,0.0,0.0
1,2025-01-01T00:14:47.000Z,2025-01-01T00:16:15.000Z,0,0.4,1,N,170,170,1,4.4,3.5,0.5,2.35,0.0,1.0,11.75,2.5,0.0,0.0
1,2025-01-01T00:39:27.000Z,2025-01-01T00:51:51.000Z,0,1.6,1,N,234,148,1,12.1,3.5,0.5,2.0,0.0,1.0,19.1,2.5,0.0,0.0
1,2025-01-01T00:53:43.000Z,2025-01-01T01:13:23.000Z,0,2.8,1,N,148,170,1,19.1,3.5,0.5,3.0,0.0,1.0,27.1,2.5,0.0,0.0
2,2025-01-01T00:00:02.000Z,2025-01-01T00:09:36.000Z,1,1.71,1,N,237,262,2,11.4,1.0,0.5,0.0,0.0,1.0,16.4,2.5,0.0,0.0


In [0]:
df.printSchema()
print("Row count:", df.count())

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

Row count: 3475226


In [0]:
from pyspark.sql import functions as F

cleaned = (
    df
    .withColumn(
        "trip_duration_minutes",
        (
            F.unix_timestamp("tpep_dropoff_datetime")
            - F.unix_timestamp("tpep_pickup_datetime")
        ) / 60
    )
    .filter(F.col("tpep_pickup_datetime") >= "2025-01-01")
    .filter(F.col("tpep_pickup_datetime") < "2025-02-01")
    .filter(F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("trip_distance") <= 100)
    .filter(F.col("trip_duration_minutes") >= 1)
    .filter(F.col("trip_duration_minutes") <= 240)
    .filter(F.col("fare_amount") >= 0)
    .filter(F.col("fare_amount") <= 500)
    .filter(F.col("total_amount") >= 0)
    .filter(F.col("total_amount") <= 600)
)

In [0]:
cleaned = (
    cleaned
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_weekday", F.dayofweek("tpep_pickup_datetime"))
)

In [0]:
print("Cleaned row count:", cleaned.count())

Cleaned row count: 3242213


In [0]:
hourly = (
    cleaned
    .groupBy("pickup_hour")
    .count()
    .orderBy("pickup_hour")
)

display(hourly)

pickup_hour,count
0,86218
1,59283
2,40029
3,25383
4,17772
5,20359
6,46405
7,96445
8,133386
9,136758


In [0]:
distance_grouped = (
    cleaned
    .withColumn(
        "distance_group",
        F.when(F.col("trip_distance") <= 1, "0-1")
        .when(F.col("trip_distance") <= 3, "1-3")
        .when(F.col("trip_distance") <= 5, "3-5")
        .when(F.col("trip_distance") <= 10, "5-10")
        .otherwise("10+")
    )
    .groupBy("distance_group")
    .agg(F.avg("fare_amount").alias("avg_fare"))
)

display(distance_grouped)

distance_group,avg_fare
1-3,13.018665732298475
0-1,7.670517062045079
3-5,21.895042005938173
5-10,33.306397808771386
10+,63.851025824455036


In [0]:
cleaned.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("yellow_taxi_january_2025_clean")

In [0]:
spark.table("yellow_taxi_january_2025_clean").count()

3242213